# Reto Semana 6: Validador de Codigos con Expresiones Regulares

## Programacion para Ciencia de Datos
**Instituto Politecnico Nacional** | Semestre Febrero-Julio 2026

---

## Contexto del Problema

Una empresa de logistica necesita un sistema para **validar codigos de productos, envios y empleados**. Actualmente, el proceso de validacion es manual y propenso a errores.

Tu tarea es crear un **validador automatico** usando expresiones regulares que verifique diferentes tipos de codigos segun sus formatos especificos.

```
╔════════════════════════════════════════════════════════════════════════════╗
║                    VALIDADOR DE CODIGOS                                    ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║   ENTRADA (stdin)                                  SALIDA (stdout)         ║
║   ───────────────                                  ──────────────          ║
║                                                                            ║
║   TEC-0001-MX          ┌──────────────┐   TEC-0001-MX,producto,VALIDO     ║
║   tec-0001-MX     ───> │  Expresiones │   tec-0001-MX,producto,INVALIDO   ║
║   ENV-2024-03-15-001234│  Regulares   │   ENV-2024-03-15-001234,...,VALIDO ║
║   EMP-VEN-0123         └──────────────┘   EMP-VEN-0123,empleado,INVALIDO  ║
║   XXX-1234                                XXX-1234,desconocido,INVALIDO    ║
║                                                                            ║
║   python main.py < codigos.txt > resultados.csv                            ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
```

## Especificaciones de Formatos

### 1. Codigo de Producto
```
Formato: ABC-1234-MX
         ───  ────  ──
          │    │    └── Pais: 2 letras mayusculas
          │    └─────── Numero: exactamente 4 digitos
          └──────────── Categoria: exactamente 3 letras mayusculas

Ejemplos validos:   TEC-0001-MX, ALI-9999-US, ROB-1234-CA
Ejemplos invalidos: tec-0001-MX, TEC-001-MX, TEC-12345-MX, TEC-0001-M
```

### 2. Codigo de Envio
```
Formato: ENV-YYYY-MM-DD-NNNNNN
         ───  ────  ──  ──  ──────
          │    │    │   │    └── Numero secuencial: 6 digitos
          │    │    │   └─────── Dia: 01-31
          │    │    └──────────── Mes: 01-12
          │    └───────────────── Año: 4 digitos (2020-2030)
          └────────────────────── Prefijo: siempre "ENV"

Ejemplos validos:   ENV-2024-03-15-001234, ENV-2025-12-01-999999
Ejemplos invalidos: ENV-2019-03-15-001234, ENV-2024-13-15-001234, ENV-2024-03-32-001234
```

### 3. Codigo de Empleado
```
Formato: EMP-XXX-NNNN
         ───  ───  ────
          │    │    └── Numero: 4 digitos (no puede empezar con 0)
          │    └─────── Departamento: 3 letras mayusculas
          └──────────── Prefijo: siempre "EMP"

Departamentos validos: VEN (Ventas), ADM (Administracion), TEC (Tecnologia),
                       LOG (Logistica), RHH (Recursos Humanos)

Ejemplos validos:   EMP-VEN-1234, EMP-TEC-9999, EMP-ADM-1000
Ejemplos invalidos: EMP-VEN-0123, EMP-XXX-1234, EMP-VEN-123
```

### 4. Codigo de Factura
```
Formato: FAC-S-NNNNNN
         ───  ─  ──────
          │   │    └── Numero: 6 digitos
          │   └─────── Serie: A, B, C, D o E
          └──────────── Prefijo: siempre "FAC"

Ejemplos validos:   FAC-A-123456, FAC-E-000001
Ejemplos invalidos: FAC-F-123456, FAC-A-12345, FAC-a-123456
```

---

## Especificacion de Entrada

### Fuente de Datos
- Los codigos se leen desde **stdin** (entrada estandar)
- **Un codigo por linea**
- Se procesan linea por linea hasta el fin del archivo (EOF)
- Las lineas vacias deben ignorarse

### Ejemplo de Entrada
```
TEC-0001-MX
ALI-9999-US
tec-0001-MX
ENV-2024-03-15-001234
ENV-2019-03-15-001234
EMP-VEN-1234
EMP-VEN-0123
FAC-A-123456
FAC-F-123456
XXX-1234
RANDOM-CODE
```

---

## Especificacion de Salida

### Formato
- **CSV** escrito a **stdout**
- Primera linea: encabezados
- Una linea por cada codigo de la entrada
- Orden: mismo orden que la entrada

### Columnas de Salida

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| `codigo` | texto | Codigo original tal como se recibio |
| `tipo` | texto | `producto`, `envio`, `empleado`, `factura` o `desconocido` |
| `valido` | texto | `VALIDO` o `INVALIDO` |

### Reglas de Deteccion de Tipo

El tipo se detecta por el **prefijo** del codigo:

| Prefijo | Tipo |
|---------|------|
| 3 letras mayusculas + `-` + 4 digitos + `-` | `producto` (cualquier combinacion de 3 letras, no solo TEC/ALI/ROB) |
| `ENV-` | `envio` |
| `EMP-` | `empleado` |
| `FAC-` | `factura` |
| Cualquier otro | `desconocido` |

**IMPORTANTE**: Un codigo puede ser del **tipo correcto pero invalido**. Por ejemplo:
- `tec-0001-MX` es tipo `producto` (tiene la estructura general) pero es **INVALIDO** porque las letras de categoria deben ser mayusculas
- `EMP-XXX-1234` es tipo `empleado` pero es **INVALIDO** porque XXX no es un departamento valido
- Un codigo `desconocido` siempre es **INVALIDO**

---

## Ejemplo Completo

### Entrada (`codigos.txt`):
```
TEC-0001-MX
ALI-9999-US
ROB-1234-CA
tec-0001-MX
TEC-001-MX
TECH-0001-MX
ENV-2024-03-15-001234
ENV-2025-12-01-999999
ENV-2019-03-15-001234
ENV-2024-13-15-001234
ENV-2024-03-32-001234
EMP-VEN-1234
EMP-TEC-9999
EMP-ADM-1000
EMP-VEN-0123
EMP-XXX-1234
EMP-VEN-123
FAC-A-123456
FAC-E-000001
FAC-B-999999
FAC-F-123456
FAC-A-12345
FAC-a-123456
XXX-1234
RANDOM-CODE
```

### Ejecucion:
```bash
python main.py < codigos.txt
```

### Salida esperada:
```
codigo,tipo,valido
TEC-0001-MX,producto,VALIDO
ALI-9999-US,producto,VALIDO
ROB-1234-CA,producto,VALIDO
tec-0001-MX,producto,INVALIDO
TEC-001-MX,desconocido,INVALIDO
TECH-0001-MX,desconocido,INVALIDO
ENV-2024-03-15-001234,envio,VALIDO
ENV-2025-12-01-999999,envio,VALIDO
ENV-2019-03-15-001234,envio,INVALIDO
ENV-2024-13-15-001234,envio,INVALIDO
ENV-2024-03-32-001234,envio,INVALIDO
EMP-VEN-1234,empleado,VALIDO
EMP-TEC-9999,empleado,VALIDO
EMP-ADM-1000,empleado,VALIDO
EMP-VEN-0123,empleado,INVALIDO
EMP-XXX-1234,empleado,INVALIDO
EMP-VEN-123,desconocido,INVALIDO
FAC-A-123456,factura,VALIDO
FAC-E-000001,factura,VALIDO
FAC-B-999999,factura,VALIDO
FAC-F-123456,factura,INVALIDO
FAC-A-12345,desconocido,INVALIDO
FAC-a-123456,factura,INVALIDO
XXX-1234,desconocido,INVALIDO
RANDOM-CODE,desconocido,INVALIDO
```

**Nota sobre la deteccion de tipo**: Observa que:
- `TEC-001-MX` es `desconocido` (no `producto`) porque no tiene 4 digitos, asi que no matchea la estructura basica `XXX-NNNN-XX`
- `EMP-VEN-123` es `desconocido` porque no tiene 4 digitos, no matchea `EMP-XXX-NNNN`
- `tec-0001-MX` es `producto` **INVALIDO** porque matchea la estructura general (3 letras-4 digitos-2 letras) pero las letras no son mayusculas
- `FAC-a-123456` es `factura` **INVALIDO** porque matchea `FAC-X-NNNNNN` pero la serie no es mayuscula

---

## Reglas de Validacion

### Regla 1: Deteccion de Tipo

El tipo se detecta por la **estructura** del codigo usando expresiones regulares:

```python
import re

# Producto: 3 letras (may o min) - 4 digitos - 2 letras
re.match(r'^[A-Za-z]{3}-\d{4}-[A-Za-z]{2}$', codigo)

# Envio: ENV- seguido de fecha y secuencial
re.match(r'^ENV-\d{4}-\d{2}-\d{2}-\d{6}$', codigo)

# Empleado: EMP- departamento - 4 digitos
re.match(r'^EMP-[A-Za-z]{3}-\d{4}$', codigo)

# Factura: FAC- serie - 6 digitos
re.match(r'^FAC-[A-Za-z]-\d{6}$', codigo)
```

### Regla 2: Validacion Estricta

Una vez detectado el tipo, se aplican las reglas estrictas:

| Tipo | Validacion estricta |
|------|---------------------|
| `producto` | Categoria y pais deben ser **mayusculas** |
| `envio` | Año 2020-2030, mes 01-12, dia 01-31 |
| `empleado` | Departamento en lista valida (`VEN`, `ADM`, `TEC`, `LOG`, `RHH`), numero **no empieza con 0** |
| `factura` | Serie en mayuscula (`A`-`E`) |
| `desconocido` | Siempre `INVALIDO` |

### Regla 3: Lineas Vacias

Las lineas vacias o con solo espacios deben **ignorarse** (no producen linea de salida).

---

## Guia de Implementacion

### Estructura Sugerida para `main.py`

```python
import sys
import re

DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']


def detectar_tipo(codigo):
    """Detecta el tipo de codigo por su estructura."""
    # TODO: Usar re.match para detectar el tipo
    pass


def validar_producto(codigo):
    """Valida que categoria y pais sean mayusculas."""
    # TODO: Implementar con regex
    pass


def validar_envio(codigo):
    """Valida rangos de fecha (año 2020-2030, mes 01-12, dia 01-31)."""
    # TODO: Implementar con regex + validacion de rangos
    pass


def validar_empleado(codigo):
    """Valida departamento valido y numero no empieza con 0."""
    # TODO: Implementar
    pass


def validar_factura(codigo):
    """Valida serie A-E en mayuscula."""
    # TODO: Implementar
    pass


def validar_codigo(codigo):
    """Detecta tipo y valida. Retorna (tipo, es_valido)."""
    tipo = detectar_tipo(codigo)
    if tipo == "producto":
        return tipo, validar_producto(codigo)
    elif tipo == "envio":
        return tipo, validar_envio(codigo)
    elif tipo == "empleado":
        return tipo, validar_empleado(codigo)
    elif tipo == "factura":
        return tipo, validar_factura(codigo)
    else:
        return "desconocido", False


def main():
    print("codigo,tipo,valido")
    for linea in sys.stdin:
        codigo = linea.strip()
        if not codigo:
            continue
        tipo, es_valido = validar_codigo(codigo)
        print(f"{codigo},{tipo},{'VALIDO' if es_valido else 'INVALIDO'}")


if __name__ == "__main__":
    main()
```

---

## Conceptos Clave: Expresiones Regulares en Python

In [ ]:
import re

# re.match() intenta hacer match al INICIO del string
# Usa ^ y $ para asegurar match completo

# Ejemplo: validar exactamente 3 letras mayusculas
patron = r'^[A-Z]{3}$'
print(re.match(patron, "ABC"))   # Match
print(re.match(patron, "AB"))    # None (solo 2 letras)
print(re.match(patron, "abc"))   # None (minusculas)
print(re.match(patron, "ABCD"))  # None (4 letras)

In [ ]:
# Grupos de captura: extraer partes del match
patron_producto = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'

m = re.match(patron_producto, "TEC-0001-MX")
if m:
    print(f"Categoria: {m.group(1)}")  # TEC
    print(f"Numero: {m.group(2)}")     # 0001
    print(f"Pais: {m.group(3)}")       # MX

# Si no matchea, re.match retorna None
m2 = re.match(patron_producto, "tec-0001-MX")
print(f"\ntec-0001-MX matchea patron estricto? {m2 is not None}")

# Pero si matchea un patron mas flexible (para detectar tipo)
patron_flexible = r'^[A-Za-z]{3}-\d{4}-[A-Za-z]{2}$'
m3 = re.match(patron_flexible, "tec-0001-MX")
print(f"tec-0001-MX matchea patron flexible? {m3 is not None}")

In [ ]:
# Validar rangos numericos despues del match regex
# Ejemplo: validar que el mes sea 01-12

codigo_envio = "ENV-2024-03-15-001234"
patron_envio = r'^ENV-(\d{4})-(\d{2})-(\d{2})-(\d{6})$'

m = re.match(patron_envio, codigo_envio)
if m:
    anio = int(m.group(1))
    mes = int(m.group(2))
    dia = int(m.group(3))
    
    print(f"Año: {anio}, Mes: {mes}, Dia: {dia}")
    print(f"Año valido (2020-2030): {2020 <= anio <= 2030}")
    print(f"Mes valido (1-12): {1 <= mes <= 12}")
    print(f"Dia valido (1-31): {1 <= dia <= 31}")

---

## Como Ejecutar y Probar

### 1. Crear archivo de prueba

Crea un archivo `codigos.txt` con los codigos de la seccion de ejemplo.

### 2. Ejecutar el programa

```bash
# Linux/Mac
python main.py < codigos.txt

# Windows (PowerShell)
Get-Content codigos.txt | python main.py

# Windows (CMD)
type codigos.txt | python main.py
```

### 3. Comparar con la salida esperada

```bash
# Guardar tu salida
python main.py < codigos.txt > mi_salida.txt

# Comparar
diff mi_salida.txt salida_esperada.txt
```

### Como se Evalua Automaticamente

```bash
python main.py < caso_prueba.txt > tu_salida.txt
diff tu_salida.txt salida_esperada.txt
# Si son identicas = CORRECTO
```

**IMPORTANTE**: La salida debe ser **EXACTA**. Sin espacios extra, sin mensajes adicionales, sin emojis.

---

## Tabla de Casos de Prueba

| # | Codigo | Tipo | Valido | Razon |
|---|--------|------|--------|-------|
| 1 | `TEC-0001-MX` | producto | VALIDO | Formato correcto |
| 2 | `ALI-9999-US` | producto | VALIDO | Formato correcto |
| 3 | `tec-0001-MX` | producto | INVALIDO | Minusculas en categoria |
| 4 | `TEC-001-MX` | desconocido | INVALIDO | Solo 3 digitos, no matchea estructura |
| 5 | `TECH-0001-MX` | desconocido | INVALIDO | 4 letras en categoria |
| 6 | `ENV-2024-03-15-001234` | envio | VALIDO | Fecha y formato correctos |
| 7 | `ENV-2019-03-15-001234` | envio | INVALIDO | Año fuera de rango (2020-2030) |
| 8 | `ENV-2024-13-15-001234` | envio | INVALIDO | Mes 13 no existe |
| 9 | `ENV-2024-03-32-001234` | envio | INVALIDO | Dia 32 no existe |
| 10 | `EMP-VEN-1234` | empleado | VALIDO | Departamento valido, numero OK |
| 11 | `EMP-VEN-0123` | empleado | INVALIDO | Numero empieza con 0 |
| 12 | `EMP-XXX-1234` | empleado | INVALIDO | Departamento no valido |
| 13 | `FAC-A-123456` | factura | VALIDO | Serie y numero correctos |
| 14 | `FAC-F-123456` | factura | INVALIDO | Serie F no existe |
| 15 | `FAC-a-123456` | factura | INVALIDO | Serie en minuscula |
| 16 | `XXX-1234` | desconocido | INVALIDO | No matchea ningun formato |

---

## Errores Comunes

### 1. No usar raw strings para regex
```python
# INCORRECTO - \d se interpreta como escape
patron = "^[A-Z]{3}-\d{4}$"

# CORRECTO - raw string
patron = r'^[A-Z]{3}-\d{4}$'
```

### 2. Olvidar anclar con ^ y $
```python
# INCORRECTO - matchea parcialmente
re.match(r'ENV-\d{4}', "ENV-2024-XX")  # Match! (solo valida el inicio)

# CORRECTO - match completo
re.match(r'^ENV-\d{4}-\d{2}-\d{2}-\d{6}$', "ENV-2024-XX")  # None
```

### 3. No validar rangos despues del regex
```python
# El regex acepta ENV-2024-99-99-000000 (mes 99, dia 99)
# Debes validar rangos DESPUES del match:
m = re.match(r'^ENV-(\d{4})-(\d{2})-(\d{2})-(\d{6})$', codigo)
if m:
    mes = int(m.group(2))
    if not (1 <= mes <= 12):
        return False  # Mes invalido
```

### 4. Imprimir mensajes extra
```python
# INCORRECTO
print("Procesando codigo...")
print(f"{codigo},{tipo},{valido}")

# CORRECTO - solo la salida CSV
print(f"{codigo},{tipo},{valido}")
```

---

## Entregables

### Repositorio en GitHub

```
reto-semana-06/
├── README.md           # Documentacion
├── main.py             # Tu solucion (lee de stdin, escribe a stdout)
├── .gitignore          # Archivos a ignorar
└── tests/              # (Opcional)
    ├── codigos.txt
    └── salida_esperada.txt
```

### Requisitos

- **main.py**: Lee codigos de stdin (uno por linea), escribe CSV a stdout
- **README.md**: Instrucciones de uso, ejemplo
- **Minimo 3 commits** significativos
- Usar **expresiones regulares** (`import re`) para la validacion

---

## Criterios de Evaluacion

| Criterio | Puntos |
|----------|--------|
| Deteccion correcta del tipo de codigo | 20 |
| Validacion de producto (regex + mayusculas) | 15 |
| Validacion de envio (regex + rangos de fecha) | 15 |
| Validacion de empleado (regex + departamento + numero) | 15 |
| Validacion de factura (regex + serie valida) | 15 |
| Formato de salida exacto (CSV) | 10 |
| Codigo limpio y organizado con funciones | 5 |
| Git (commits, README) | 5 |
| **Total** | **100** |

---

## Consejos

- Usa [regex101.com](https://regex101.com/) para probar tus expresiones regulares
- Recuerda usar raw strings (`r'...'`) para los patrones
- Usa grupos de captura `()` para extraer partes del codigo
- Primero detecta el tipo, luego valida con reglas estrictas
- Primero haz que funcione, luego optimiza

---

## Fecha de Entrega

**Viernes de la Semana 6, 23:59 hrs**

---

*Reto Semana 6 - Programacion para Ciencia de Datos - IPN 2026*